# Evaluating the Fine-Tuned RoBERTa Model on Zoning By-laws

### Introduction

**This experiement aims to evaluate the fine-tuned and pre-trained RoBERTa QA model to extract information from Zoning By-laws.** The model being evaluated is the one fine-tuned in the [Fine-Tuning RoBERTa on Zoning By-laws Experiment](https://github.com/JoT8ng/zoning-extraction-pipelines/blob/main/finetuning_pipeline/finetuning_qa_experiment.ipynb).

### Evaluation Metric

The Stanford Question Answering Dataset (SQuAD) is a widely used benchmark dataset for evaluating question answering models. The original paper (Rajpurkar et al., 2016) introduced two key evaluation metrics that have since become standard in the field.

* **Exact Match (EM):** This metric measures the percentage of questions where the model's answer exactly matches one of the ground truth answers.
* **F1 Score:** This metric calculates the overlap between the predicted answer and the ground truth answers. It considers both precision (the number of correct answers provided by the model) and recall (the number of correct answers that should have been provided). The F1 score is the harmonic mean of precision and recall, providing a balance between the two. A higher F1 score indicates a better performing model. The F1 score is good with imbalanced datasets where accuracy can be misleading. [More information](https://www.geeksforgeeks.org/machine-learning/f1-score-in-machine-learning/)

Hugging Face's evaluate library provides allows you to compute exact Mmatch (EM) and token-level F1.

An evaluation CSV dataset of 100 samples will be used to evaluate the model. Unlike [Zero shot QA experiment 2](https://github.com/JoT8ng/zoning-extraction-pipelines/blob/main/zeroshot_qa/zeroshot_qa_experiment2.ipynb), some of the contexts will be longer to emulate how the model will be used in realistic scenarios. Zoning by-laws are long documents so users may put in longer contexts. A tokenizer will be used with a sliding window to handle longer contexts in the evaluation dataset.

### Evaluation and Metrics

In [7]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
import pandas as pd
import evaluate

# Load SQuAD metrics
squad_metric = evaluate.load("squad")

# Set up array to store model responses
results = []

# Evaluation helper function to prepare inputs for Hugging Face SQuAD metrics
def evaluate_model(res, model):

    # res or results: results dictionary containing the outputs of the predictions and ground truth

    predictions = []
    references = []

    for r in res:
        predictions.append({
            "id": str(r["doc_id"]),
            "prediction_text": r[model]
        })
        references.append({
            "id": str(r["doc_id"]),
            "answers": {
                "text": [r["ground_truth"]],
                "answer_start": [0]  # dummy value
            }
        })

    # Compute metrics
    return squad_metric.compute(predictions=predictions, references=references)

# Load the the evaluation dataset
dataset = pd.read_csv("EvaluationDataset.csv")

# Load Fine-Tuned Model
model_path = "./roberta-zoning-qa-finetuned1"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForQuestionAnswering.from_pretrained(model_path)

# Load RoBERTa base squad 2
roberta_qa = pipeline(
    "question-answering",
    model = "deepset/roberta-base-squad2"
)

# Create QA pipeline with sliding window
roberta_zoning_qa = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    stride=128,
    truncation="only-second"
)

# Run QA
for _, data in dataset.iterrows():
    q = data['question']
    ctext = data['context']
    truth = data['ground_truth']

    finetuned_response = roberta_zoning_qa(question=q, context=ctext)['answer']
    base_response = roberta_qa(question=q, context=ctext)['answer']

    results.append({
        "doc_id": data['doc_id'],
        "question": q,
        "ground_truth": truth,
        "municipality": data['municipality'],
        "finetune_answer": finetuned_response,
        "base_answer": base_response
    })

dataframe = pd.DataFrame(results)
dataframe.to_csv("roberta-zoning-qa-results-1.csv", index = True)
dataframe

Device set to use cpu
Device set to use cpu


,doc_id,question,ground_truth,municipality,finetune_answer,base_answer
0,1,What is the maximum building height for access...,4.0 m | 1 storey,Burnaby,4.0 m | 1 storey,4.0 m
1,2,What is the minimum lot frontage requirements?,6 metres (19.7 ft.),Niagara Falls,6 metres (19.7 ft.).,6 metres (19.7 ft.).
2,3,Where does a child care facility in the R1 dis...,on a corner lot,Burnaby,on a corner lot,a corner lot
3,4,For lots in the R1 district on the Community H...,2.0 m,Burnaby,2.0 m,2.0 m
4,5,What is the minimum rear yard depth where any ...,10 metres (33.0 ft.) plus any applicable dista...,Niagara Falls,10 metres (33.0 ft.),10 metres
...,...,...,...,...,...,...
95,96,What is the Rear Yard minimum yards for Privat...,see Section 5.7,St. Catharine,Section 5.7,Section 5.7
96,97,What is the Min. Density Per Hectare for Dwell...,85 Units,St. Catharine,85 Units,85 Units
97,98,"What is the Min. Lot Frontage for Dwelling, To...",6 m,St. Catharine,6 m,6 m
98,99,What is the Min. Landscaped Open Space for Dwe...,25%,St. Catharine,25%.,25%.


In [8]:
# Evaluation and metrics

metrics = evaluate_model(results, "finetune_answer")
base_metrics = evaluate_model(results, "base_answer" )

print("Fine-Tuned Model Metrics:", metrics)
print("RoBERTa base Metrics:", base_metrics)

Fine-Tuned Model Metrics: {'exact_match': 49.0, 'f1': 71.10281743548464}
RoBERTa base Metrics: {'exact_match': 43.0, 'f1': 66.86070487829579}
